<div style="font-size: 0.85em; line-height: 1.5;">

<h3>HyDE – Hypothetical Document Embeddings</h3>

<p><strong>What is it?</strong><br>
HyDE uses an LLM to generate a hypothetical answer to the user’s question, then embeds that answer and uses it as the search query. This often retrieves more relevant documents for broad or vague questions.</p>

<p><strong>How it works</strong></p>
<ol>
  <li>User asks a question.</li>
  <li>LLM generates a detailed, hypothetical answer (even if not perfectly correct).</li>
  <li>The hypothetical answer is embedded.</li>
  <li>The vector store is searched using that embedding.</li>
  <li>Retrieved chunks are passed to the LLM to produce the final answer.</li>
</ol>

<p><strong>Why use it?</strong></p>
<ul>
  <li><strong>Improves recall</strong> – broad questions become detailed search queries.</li>
  <li><strong>Reduces vocabulary mismatch</strong> – generated text contains terms that match documents.</li>
  <li><strong>Works well with dense retrieval</strong> – combines LLM generation with embedding search.</li>
</ul>

<p><strong>Visualisation</strong></p>
<pre>
User Query
   │
   ▼
[LLM] Generate hypothetical answer
   │
   ▼
Embed hypothetical answer
   │
   ▼
Search vector store
   │
   ▼
Retrieve relevant chunks
   │
   ▼
Final LLM answer
</pre>

<p><strong>Implementation</strong><br>
We will use a prompt to generate a hypothetical document, embed it, and use that embedding for retrieval.</p>

</div>

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader, BSHTMLLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chains import HypotheticalDocumentEmbedder
#from langchain.retrievers.hyde import HyDE
from dotenv import load_dotenv

In [2]:
# Load relevant documents
files = [
    ('../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/crop_disease.pdf', 'pdf'),
    ('../../04_data_ingestion_document_processing/data/agriculture.html', 'html'),
    ('../../04_data_ingestion_document_processing/data/agriculture.txt', 'txt'),
]

all_docs = []
for path, ftype in files:
    if ftype == 'pdf':
        loader = PyPDFLoader(path)
    elif ftype == 'html':
        loader = BSHTMLLoader(path, open_encoding='utf-8', bs_kwargs={'features': 'html.parser'})
    elif ftype == 'txt':
        loader = TextLoader(path, encoding='utf-8')
    else:
        continue
    all_docs.extend(loader.load())

# Split and add metadata
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(all_docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get('source', '')
    file_name = source.split('\\')[-1] if '\\' in source else source.split('/')[-1]
    chunk.metadata['file_name'] = file_name
    chunk.metadata['doc_type'] = (
        'pdf' if file_name.endswith('.pdf')
        else 'html' if file_name.endswith('.html')
        else 'txt'
    )
    chunk.metadata['language'] = 'English'
    chunk.metadata['chunk_id'] = f'{file_name}_{i+1:03d}'

print(f'Loaded {len(all_docs)} docs, created {len(chunks)} chunks.')

Loaded 37 docs, created 185 chunks.


In [3]:
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=None
)

print('✅ Vector store ready.')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Vector store ready.


Step 4: Create the HypotheticalDocumentEmbedder

In [4]:
# LLM for generating hypothetical answer
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

hyde_embedder = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key='web_search'
)

print(' HypotheticalDocumentEmbedder ready.')

 HypotheticalDocumentEmbedder ready.


Step 5: Use HyDE to Retrieve Documents

In [5]:
# Broad question that previously failed
query = 'What are the major diseases affecting crops and livestock in Nigeria, and how are they managed?'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved 5 documents using HyDE:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected 
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. Agriculture in Nigeria: Crops, Livestock, and Disease Management



Agriculture in Nigeria
Last updated: August 2026


Introduction
   Source: ../../04_data_ingestion_document_pro

In [6]:
# Broad question that previously failed
query = 'What are the main problems affecting health and farming in Nigeria?'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Retrieved 5 documents using HyDE:

1. major health problem in Nigeria.  
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, cancer, meningitis, stroke and 
tuberculosis. 
 Ma
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
2. tackle the major public health issues in the     
country (i.e. HIV/AIDs, Tuberculosis and      
Malaria remain major health issues in Nigeria) 
and it also try more on the incidence of drug   
resistant TB and extremely drug resistant TB in 
order t
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
3. scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Heal

In [7]:
# Broad question that previously failed
query = 'What diseases affect animals and plants, and how can farmers deal with them?'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Retrieved 5 documents using HyDE:

1. Use disease-resistant seeds and breeds.
Practise crop rotation and clean farming.
Ensure proper nutrition and clean water for animals.
Maintain hygiene in farms and pens.
Quarantine new plants or animals before introducing them.
Consult extension off
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. • The major factors responsible for crop 
diseases are fungi, bacteria, viruses and 
nematodes. 
• Others are nutrient deficiencies and air 
pollutants. 
• The severity of crop diseases can range 
from mild leaf or fruit damage to death.
   Source: ../../04_data_ingestion_document_processing/data/crop_disease.pdf
------------------------------------------------------------
3. Introduction
O Crop diseases are those conditions 
when pathogens, lack or inadequacy 
of nutrient supply and various other 
factors of the environment cause 
disruptions or hinderances to th

In [8]:
# Broad question that previously failed
query = 'Tell me about issues in Nigeria’s food and health sectors.'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Retrieved 5 documents using HyDE:

1. cultural and religious beliefs and practices. Health      
problems in Nigeria are challenging, but addressing them 
using public health principles is necessary to support   
stability in this important area of the world. 
The inadequate programs des
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
2. publications were reviewed with emphasis on the public 
health issues in Nigeria. 
 
Results and Discussion 
Nigeria faces many public health problems and         
challenges.15 The health issues that Nigeria faces are 
infectious diseases, sewage di
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
3. scientific database sources, web search engines, direct observation and relevant documents from the Nigerian Ministry 
of Heal

In [10]:
# Broad question that previously failed
query = 'list top 10 causes of death in Nigeria and list Common Crop Diseases and Control'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Retrieved 5 documents using HyDE:

1. major health problem in Nigeria.  
 The top causes of death in Nigeria are; malaria, 
lower respiratory infections, HIV/AIDS,      
diarrheal diseases, road injuries, protein -energy 
malnutrition, cancer, meningitis, stroke and 
tuberculosis. 
 Ma
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
2. diseases, alcohol abuse, environment degradation, road 
traffic injuries etcetera.5 
 
The top 10 causes of death in Nigeria are as follows:3 
 
 Malaria (20%) 
 Lower Respiratory Infection (19%) 
 HIV/AIDS (9%) 
 Diarrheal Diseases (5%) 
 Road 
   Source: ../../04_data_ingestion_document_processing/data/nigeria_health_diseases_and_prevention.pdf
------------------------------------------------------------
3. like Hypertension, Cancer, Obesity etc. Most of the 
Nigerians (young and old) die of different             
preventable disea

In [11]:
# Broad question that previously failed
query = 'List Common Crop Diseases and Control, also list top 10 causes of death in Nigeria'

# Generate hypothetical embedding
hyde_embedding = hyde_embedder.embed_query(query)

# Retrieve documents using the hypothetical embedding
docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=5)

print(f'Retrieved {len(docs)} documents using HyDE:\n')

for i, doc in enumerate(docs, start=1):
    print(f'{i}. {doc.page_content[:250]}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)


Retrieved 5 documents using HyDE:

1. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infecte
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
------------------------------------------------------------
2. Common Crop Diseases and Control

1. Cassava Mosaic Disease
   Affected crop: Cassava
   Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
   Control: Use disease-free cuttings, plant resistant varieties, and remove infected 
   Source: ../../04_data_ingestion_document_processing/data/agriculture.txt
------------------------------------------------------------
3. • The major factors responsible for crop 
diseases are fungi, bacteria, viruses and 
nematodes. 
• Others are nutrient deficiencies and air 
pollutants. 
• The severity of crop di

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt that asks the LLM to break the original question into separate sub-questions
decompose_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are an AI assistant. Break the user question into 2-4 simpler sub-questions. Output each sub-question on a new line. Do not answer them.'),
    ('human', '{question}')
])

# Create a chain that returns sub-questions as a single string
decompose_chain = decompose_prompt | llm | StrOutputParser()



Define a Helper to Decompose and Retrieve

In [13]:
def decompose_and_retrieve(question, top_k=4):
    """Break a compound question into sub-questions, retrieve for each, and combine."""
    # Generate sub-questions
    sub_questions_text = decompose_chain.invoke({'question': question})
    sub_questions = [q.strip() for q in sub_questions_text.split('\n') if q.strip()]
    print(f'Sub-questions:\n{sub_questions}\n')
    
    # Retrieve documents for each sub-question using HyDE
    all_docs = []
    seen = set()
    for sub_q in sub_questions:
        # Generate hypothetical embedding for this sub-question
        hyde_embedding = hyde_embedder.embed_query(sub_q)
        # Search vector store
        docs = vectorstore.similarity_search_by_vector(hyde_embedding, k=top_k)
        # Add unique docs
        for doc in docs:
            key = doc.page_content.strip()
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)
    
    return all_docs, sub_questions

In [16]:
question = 'List Common Crop Diseases and Control, also list top 10 causes of death in Nigeria'

combined_docs, sub_qs = decompose_and_retrieve(question, top_k=5)

print(f'Combined top {len(combined_docs)} unique documents:\n')
for i, doc in enumerate(combined_docs, start=1):
    print(f'{i}. {doc.page_content}')
    print(f'   Source: {doc.metadata.get("source", "unknown")}')
    print('-' * 60)

Sub-questions:
['What are some common crop diseases?', 'What are the control methods for these crop diseases?', 'What are the top 10 causes of death in Nigeria?', 'Can you provide details or statistics on these causes of death?']

Combined top 14 unique documents:

1. • The major factors responsible for crop 
diseases are fungi, bacteria, viruses and 
nematodes. 
• Others are nutrient deficiencies and air 
pollutants. 
• The severity of crop diseases can range 
from mild leaf or fruit damage to death.
   Source: ../../04_data_ingestion_document_processing/data/crop_disease.pdf
------------------------------------------------------------
2. Common Crop Diseases and Their Control

1. Cassava Mosaic Disease
Affected Crop: Cassava
Symptoms: Yellowing and mottling of leaves, stunted growth, reduced yield.
Control/Cure: Use disease-free cuttings, plant resistant varieties, and remove infected plants early.
   Source: ../../04_data_ingestion_document_processing/data/agriculture.html
---------